# Suite EVL — Evaluators

`Evaluators.OfAny(expression, scope)` computes an expression's value with the variables in `scope` bound, and
`Evaluators.OfLiteral`, `OfOperation`, `OfVariable` and `OfLet` evaluate one kind. Evaluation is three-valued (absent
and incomparable values are unknown, and logic follows Kleene), never coerces native types, and evaluates only the core
operations.

In [ ]:
import { Evaluators as V, Expressions as E, Proxies, Schemas as S } from "../src/Framework/index.js";
import { KeyError, NotImplementedError, ValueError } from "../src/Framework/Errors.js";
import { assert, raises, same, text } from "./support.js";

const op = new E.OfOperation.Builder().name("f").arguments(1n, "x").create();
const self = E.variable("this");

/** Evaluates `expression` with `scope` bound. */
const ev = (expression: E.OfAny.Spec, scope: Record<string, unknown> = {}) => V.OfAny(expression, scope);

## EVL-01 · Evaluating literals, variables, lets, `get` and `has`

In [ ]:
const Person = new S.OfObject.Builder().properties(text("name"), text("age", BigInt)).create();
Proxies.register("Person", Person);
const ann = (Proxies.Builders as any).Person().name("Ann").age(30n).create();
const nobody = (Proxies.Builders as any).Person().create();
assert(ev(3n) === 3n && ev(E.literal("x")) === "x" && ev(self, { this: ann }) === ann);
assert(ev(self.age, { this: ann }) === 30n && ev(self.age, { this: nobody }) === null); // absent: unknown
assert(ev(self.has("age"), { this: ann }) === true && ev(self.has("age"), { this: nobody }) === false);
assert(ev(self.age, { this: null }) === null && ev(self.has("age"), { this: null }) === null); // unknown object
assert(ev(E.let_("a", self.age, E.variable("a").add(1n)), { this: ann }) === 31n);
assert(ev(E.let_("a", 1n, E.let_("a", 2n, E.variable("a")))) === 2n); // the innermost binding wins
assert(ev(E.let_("a", 1n, E.variable("a").add(E.let_("a", 2n, E.variable("a"))))) === 3n); // and only in its body
assert(ev(self.kind, { this: E.OfLiteral.resolve(1n) }) === "literal"); // expressions are objects too
raises(KeyError, () => ev(E.variable("x")), "variable 'x' is not bound");
raises(TypeError, () => ev(E.operation("get", 1n, "x")), "get expects an object, got int");
raises(TypeError, () => ev(E.operation("has", self, 1n), { this: ann }), "has expects a property name, got int");

## EVL-02 · Comparisons are three-valued and never coerce

In [ ]:
const unit = E.literal(1n);
assert(same([ev(unit.lt(2n)), ev(unit.le(1n)), ev(unit.gt(2n)), ev(unit.ge(1n)), ev(unit.eq(1n)), ev(unit.ne(1n))], [
  true, true, false, true, true, false]));
assert(ev(E.literal("b").gt("a")) === true && ev(E.literal(new Uint8Array([1])).lt(new Uint8Array([2]))) === true);
assert(ev(unit.eq(1.0)) === null && ev(unit.lt(1.5)) === null && ev(unit.ne("1")) === null); // different types
assert(ev(E.literal(true).eq(false)) === false && ev(E.literal(true).lt(false)) === null); // bools are not ordered
assert(ev(E.literal(true).le(true)) === true);
const nan = E.literal(NaN);
assert(ev(nan.eq(NaN)) === true && ev(nan.eq(1.0)) === false && ev(nan.lt(1.0)) === null);
assert(ev(E.literal(-0).lt(0)) === true && ev(E.literal(-0).eq(0)) === false);
const age = E.variable("x").age;
assert(ev(age.eq(1n), { x: nobody }) === null && ev(age.ne(1n), { x: nobody }) === null && ev(age.lt(1n), { x: nobody }) === null);
const [first, second] = [E.variable("a"), E.variable("b")];
assert(ev(first.eq(second), { a: ann, b: ann }) === true && ev(first.eq(second), { a: ann, b: nobody }) === false); // objects: by identity
assert(ev(first.lt(second), { a: ann, b: nobody }) === null && ev(first.eq(1n), { a: ann }) === null);

## EVL-03 · Logic follows Kleene; arithmetic keeps one type

In [ ]:
const [T, F, U] = [E.literal(true), E.literal(false), E.variable("u")]; // u is unknown
const table = (method: string) => [T, F, U].map((x) => [T, F, U].map((y) => ev(x[method](y), { u: null })));
assert(globalThis.JSON.stringify(table("and_")) === globalThis.JSON.stringify([[true, false, null], [false, false, false], [null, false, null]]));
assert(globalThis.JSON.stringify(table("or_")) === globalThis.JSON.stringify([[true, true, true], [true, false, null], [true, null, null]]));
assert(globalThis.JSON.stringify(table("implies")) === globalThis.JSON.stringify([[true, false, null], [true, true, true], [true, null, null]]));
assert(same([T, F, U].map((x) => ev(x.not_(), { u: null })), [false, true, null]));
const missing = E.variable("missing"); // never evaluated when the first operand decides
assert(ev(F.and_(missing)) === false && ev(T.or_(missing)) === true && ev(F.implies(missing)) === true);
raises(TypeError, () => ev(T.and_(1n)), "and expects bool operands, got int");
raises(TypeError, () => ev(E.literal("x").not_()), "not expects bool operands, got str");
const pair = E.literal(2n);
assert(same([ev(pair.add(3n)), ev(pair.sub(3n)), ev(pair.mul(3n)), ev(pair.neg())], [5n, -1n, 6n, -2n]));
assert(ev(E.literal(1.5).add(1.0)) === 2.5 && ev(E.literal(2n ** 70n).mul(2n)) === 2n ** 71n);
assert(ev(U.add(1n), { u: null }) === null && ev(U.neg(), { u: null }) === null);
raises(TypeError, () => ev(E.literal(1n).add(1.5)), "add expects numbers of one type, got int and float");
raises(TypeError, () => ev(T.sub(T)), "sub expects numbers of one type, got bool and bool");
raises(TypeError, () => ev(E.literal("x").neg()), "neg expects a number, got str");

## EVL-04 · Checking: `validate()` finds what evaluation would raise

In [ ]:
assert(same((E.operation("lt", 1n).data as E.OfOperation.Data).validate(), ["lt takes 2 arguments, got 1"]));
const foo = E.operation("foo", 1n).data as E.OfOperation.Data;
assert(foo.validate().length === 0 && same(foo.validate({ core: true }), ["'foo' is not a core operation"]));
const checked = E.let_("a", self.age, E.variable("a").ge(18n).and_(self.has("email"))).data as E.OfLet.Data;
assert(checked.validate({ bound: ["this"], core: true }).length === 0);
assert(same(checked.validate(), ["value: argument 0: variable 'this' is not bound",
  "body: argument 1: argument 0: variable 'this' is not bound"]));
raises(NotImplementedError, () => ev(foo), "'foo' is not a core operation");
raises(TypeError, () => ev(E.operation("lt", 1n)), "lt takes 2 arguments, got 1");
raises(ValueError, () => ev(new E.OfLiteral.Data()), "a literal needs a value");
raises(ValueError, () => ev(new E.OfLet.Data("x")), "a let needs a value");
raises(ValueError, () => ev(new E.OfLet.Data("x", E.OfLiteral.resolve(1n))), "a let needs a body");
const cyclic = new E.OfOperation.Builder().name("not").create();
new E.OfOperation.Builder(cyclic).arguments(cyclic).update();
raises(ValueError, () => ev(cyclic), "the expression contains a cycle");
const circular = new E.OfLet.Data("a", null, E.OfLiteral.resolve(1n));
circular.value = circular;
assert(same(circular.validate(), ["value: the expression contains a cycle"]));
raises(TypeError, () => ev(new E.OfOperation.Data("not", [7n])), "not an expression: 7");

## EVL-05 · The union's discriminators pick each kind by its tag

In [ ]:
const samples = [E.OfLiteral.resolve(1n), op, E.OfVariable.resolve("x"), E.let_("a", 1n, 2n).data];
E.OfAny.Schema.branches.forEach((branch, i) => {
  const when = branch.when as E.OfOperation.Data;
  assert(when.validate({ bound: ["this"], core: true }).length === 0);
  assert(same(samples.map((sample) => ev(when, { this: sample })), samples.map((sample) => sample === samples[i])));
});

## EVL-06 · Each kind has its own entry point, which accepts that kind's Specs

In [ ]:
const x = E.variable("x");
assert(V.OfLiteral(3n) === 3n && V.OfLiteral(E.literal("s")) === "s");
assert(V.OfVariable("x", { x: 1n }) === 1n && V.OfVariable(x, { x: 2n }) === 2n);
assert(V.OfOperation(x.add(1n), { x: 1n }) === 2n && V.OfLet(E.let_("y", 1n, E.variable("y").neg())) === -1n);
assert(V.OfAny(x, { x: null }) === null && V.OfAny((a: E.OfAny.Builder) => a.as_literal(1n)) === 1n);
raises(TypeError, () => V.OfLet(3n as any), "expected a let or a callable taking its builder, got 3");
raises(TypeError, () => V.OfOperation(x), "expected an operation or a callable taking its builder"); // a variable, not an operation